In [36]:
import pandas as pd
import numpy as np
import os

df = pd.read_excel(r"C:\Users\Admin\Desktop\P&P Round 18 pt2\QC_data\Data File_from_google_part_2.xlsx")



In [37]:
csv = pd.read_excel(r"C:\Users\Admin\Desktop\Paints Top Line Report\CSV December\Paints CSV Report - Dec'25.xlsx",sheet_name="Oct-Mar'25")
csv['MOP'] = csv[['VALUE INR (Exc. GST)','Volume']].apply(lambda x:x[0]/x[1],axis=1)
average_mop = csv.groupby('SKU')['MOP'].mean()

C:\Users\Admin\AppData\Local\Temp\ipykernel_38552\3473008543.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  csv['MOP'] = csv[['VALUE INR (Exc. GST)','Volume']].apply(lambda x:x[0]/x[1],axis=1)
C:\Users\Admin\AppData\Local\Temp\ipykernel_38552\3473008543.py:2: RuntimeWarning: invalid value encountered in scalar divide
  csv['MOP'] = csv[['VALUE INR (Exc. GST)','Volume']].apply(lambda x:x[0]/x[1],axis=1)
C:\Users\Admin\AppData\Local\Temp\ipykernel_38552\3473008543.py:2: RuntimeWarning: divide by zero encountered in scalar divide
  csv['MOP'] = csv[['VALUE INR (Exc. GST)','Volume']].apply(lambda x:x[0]/x[1],axis=1)


In [38]:
average_mop = average_mop.reset_index()

In [39]:
# 1. Extract packsize from 'Last SKU Name' (last part after the final " - ")
last_part = df['SKU Name'].astype(str).str.rsplit(' - ', n=1).str[-1]
# Extract numeric value and unit (handles "1 L", "20 L", "100 ML", "0.1 L", etc.)
packsize_match = last_part.str.extract(r'(\d+\.?\d*)\s*(ML|ml|L|l)?', expand=True)
packsize_match.columns = ['num', 'unit']
packsize_num = packsize_match['num'].astype(float)
# If unit is ML/ml: convert to L (e.g. 100 ML -> 0.1 L). Otherwise treat as L.
is_ml = packsize_match['unit'].str.upper().str.startswith('M', na=False)
df['Packsize'] = np.where(is_ml, packsize_num / 1000, packsize_num)

# 2. Calculate price per litre (price divided by packsize in L)
df['Price_Per_Litre'] = df['Fin_Price'] / df['Packsize']

# 3. Merge with average_mop to get SKU-level average (SKU Name in df matches SKU in average_mop)
df = df.merge(average_mop, left_on='SKU Name', right_on='SKU', how='left', suffixes=('', '_avg'))

# 4. Flag if price per litre is outside 30% range of SKU average MOP (only when average exists)
df['Flag'] = df['MOP'].notna() & (
    (df['Price_Per_Litre'] < df['MOP'] * 0.7) | 
    (df['Price_Per_Litre'] > df['MOP'] * 1.3)
)

# Drop the duplicate SKU column from merge if desired (optional)
df = df.drop(columns=['SKU'], errors='ignore')

# Summary: how many Flag=False because SKU match could not be found
no_sku_match = df['MOP'].isna().sum()
print(f"Transactions with no matching SKU found (Flag=False for this reason): {no_sku_match}")

df

Transactions with no matching SKU found (Flag=False for this reason): 179


,S. No.,City,Outlet Code,Outlet Name,Fin_Price,Fin_Qty,Fin_Amt,Brand Name,Product,SKU Name,File name,Packsize,Price_Per_Litre,MOP,Flag
0,40,Durgapur,51742490,Pandit Hardware ( Swastika Supplier),66.0,8,591.0,asianpaints,Acrylic Putty,asianpaints - TRUCARE ACRYLIC WALL White - Put...,output_16mar26_Assam_Guwahati_Shiva Hardware_7...,1.000,66.0,NaN,False
1,138,Kolkata Metropolitan Region,70010155,Balaji Trading Company,199.0,12,2388.0,Berger,Primer,Berger - Parrot Wood - Primer- 1 ML,output_16mar26_Chandigarh_Chandigarh_Kamal Ent...,0.001,199000.0,NaN,False
2,147,Kolkata Metropolitan Region,70010155,Balaji Trading Company,199.0,6,1338.0,Berger,Primer,Berger - Parrot Wood - Primer- 1 ML,output_16mar26_Chandigarh_Chandigarh_Kamal Ent...,0.001,199000.0,NaN,False
3,182,Kolkata Metropolitan Region,70010155,Balaji Trading Company,767.0,1,859.0,Berger,Primer,Berger - Parrot Wood - Primer- 1 ML,output_16mar26_Chandigarh_Chandigarh_Shakti Bu...,0.001,767000.0,NaN,False
4,512,Kolkata Metropolitan Region,70010461,Mahabir Hardware,199.0,6,1333.0,Berger,Primer,Berger - Parrot Wood - Primer- 1 ML,output_16mar26_Uttar Pradesh_Allahabad_Majeed ...,0.001,199000.0,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1151,1389,Kolkata Metropolitan Region,70010448,Sanskar Enterprise,3494.0,10,33796.0,Other Brands,Waterproofing,Other Brands - Waterproofing - 1 L,output_17mar26_Gujarat_Himatnagar_Patel Colour...,1.000,3494.0,NaN,False
1152,1392,Kolkata Metropolitan Region,70010448,Sanskar Enterprise,3494.0,10,33796.0,Other Brands,Waterproofing,Other Brands - Waterproofing - 1 L,output_17mar26_Gujarat_Himatnagar_Patel Colour...,1.000,3494.0,NaN,False
1153,1427,Kolkata Metropolitan Region,70010448,Sanskar Enterprise,5950.0,2,11900.0,Other Brands,Distemper,Other Brands - Distemper,output_17mar26_Gujarat_Himatnagar_Viral Engine...,NaN,NaN,61.7871,False
1154,1449,Kolkata Metropolitan Region,70010437,Standard Trading Co.,1236.0,8,9808.0,Other Brands,Interior Emulsion,Other Brands - Interior Emulsion - 4 L,output_17mar26_Jammu & Kashmir_Jammu_Mukhalya ...,4.000,309.0,NaN,False


In [40]:
df[df['Flag']==True]

,S. No.,City,Outlet Code,Outlet Name,Fin_Price,Fin_Qty,Fin_Amt,Brand Name,Product,SKU Name,File name,Packsize,Price_Per_Litre,MOP,Flag
7,752,Hyderabad Metropolitan Region,51820663,Keerthana Paints,10.0,10,102.0,asianpaints,Tools for Paints,asianpaints - Pressure Washer - Tools for Pain...,output_16mar26_Uttar Pradesh_Jhansi_Shree Maha...,1.0,10.000000,238.819391,True
10,14,Sidlaghatta,70013120,Shri Malleshwaraswamy Paints And Hardware,37.0,720,29863.0,asianpaints,Tools for Paints,asianpaints - Measuring Tape 3m - Tools for Pa...,output_16mar26_Assam_Guwahati_Laxmi Paints And...,1.0,37.000000,1112.489176,True
13,69,Durgapur,51742490,Pandit Hardware ( Swastika Supplier),166.0,10,1754.0,asianpaints,Enamel,asianpaints - Apcolite Premium Gloss C Gold - ...,output_16mar26_Chandigarh_Chandigarh_Aggarwa E...,0.2,830.000000,236.218417,True
20,135,Kolkata Metropolitan Region,70010155,Balaji Trading Company,571.0,5,3840.0,Berger,Exterior Emulsion,Berger - WeatherCoat Glow - Red Bs - Exterior ...,output_16mar26_Chandigarh_Chandigarh_Kamal Ent...,0.9,634.444444,255.433251,True
25,150,Kolkata Metropolitan Region,70010155,Balaji Trading Company,1024.0,1,1147.0,Berger,Colorants,Berger - Luxol Universal Stainers Black - Colo...,output_16mar26_Chandigarh_Chandigarh_Kamal Ent...,0.2,5120.000000,465.912703,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1071,1650,Kolkata Metropolitan Region,51818075,Rekha Paints & Hardware,4087.0,1,4653.0,Nerolac,Interior Emulsion,Nerolac - Water Based Lustre White - Enamel-10 L,output_17mar26_Madhya Pradesh_Chhindwara_Mahal...,10.0,408.700000,253.666943,True
1074,1657,Kolkata Metropolitan Region,51818075,Rekha Paints & Hardware,3010.0,1,3427.0,Nerolac,Exterior Emulsion,Nerolac - Suraksha Plus Dark Green - Exterior ...,output_17mar26_Madhya Pradesh_Chhindwara_Mahal...,4.0,752.500000,145.213623,True
1084,1678,Ranchi,51840975,Prakash Hardware,4560.0,3,12578.0,asianpaints,Primer,asianpaints - WOOD SUPREMA PINK - Primer-1 L,output_17mar26_Madhya Pradesh_Gwalior_Bhagirat...,1.0,4560.000000,188.442883,True
1096,1693,Bhiwandi,51748836,Ambika Electric And Hardware,70.0,8,632.0,asianpaints,Primer,asianpaints - TRUCARE SPARC RO REDOXIDE - Prim...,output_17mar26_Madhya Pradesh_Gwalior_Bhagirat...,1.0,70.000000,123.130011,True


In [41]:
df.to_csv(r'C:\Users\Admin\Desktop\P&P Round 18 pt2\QC_data\flagged_transactions_part_2.csv', index=False)